In [0]:
# Generate synthetic data for song_id = 99999 (deletion testing)
# These SELECT statements create dummy data that simulates a song in our tables
# but doesn't exist in the FFR API

test_song_id = 99999
test_song_name = "DUMMY_TEST_SONG"

print("="*70)
print("SYNTHETIC DATA GENERATION FOR DELETION TESTING")
print("="*70)
print(f"\nTest song ID: {test_song_id}")
print(f"Test song name: {test_song_name}")
print(f"\nGenerating SELECT statements for bronze, silver, and gold tables...\n")

# ===========================================================================
# BRONZE LAYER
# ===========================================================================

print("[BRONZE LAYER]")
print("=" * 70)

# Bronze: songlist
print("\n[1] bronze__songlist")
print("-" * 70)
df_bronze_songlist = spark.sql(f"""
SELECT 
    'Test Artist' as author,
    'https://test.com' as author_url,
    10 as difficulty,
    1 as genre,
    {test_song_id} as id,
    '180' as length,
    50 as max_nps,
    10 as min_nps,
    '{test_song_name}' as name,
    100 as note_count,
    'Test Stepper' as stepauthor,
    999999 as swf_version,
    CAST(UNIX_TIMESTAMP() as BIGINT) as timestamp,
    'TEST DATE' as timestamp_format
""")
print("✓ Synthetic songlist record generated")
display(df_bronze_songlist)

# Bronze: charts (with multiple notes)
print("\n[2] bronze__charts")
print("-" * 70)
df_bronze_charts = spark.sql(f"""
SELECT 
    array(
        array(0.0, 0, 0, 0, 1000),      -- Note 1: Right arrow at t=0
        array(0.5, 0, 0, 1000, 0),      -- Note 2: Down arrow at t=0.5
        array(1.0, 0, 1000, 0, 0),      -- Note 3: Up arrow at t=1.0
        array(1.5, 1000, 0, 0, 0),      -- Note 4: Left arrow at t=1.5
        array(2.0, 1000, 0, 0, 1000)    -- Note 5: Left+Right at t=2.0
    ) as chart,
    map('difficulty', '10', 'note_count', '5', 'name', '{test_song_name}') as info,
    {test_song_id} as song_id
""")
print("✓ Synthetic chart record generated (5 notes)")
display(df_bronze_charts)

# Bronze: playlist
print("\n[3] bronze__playlist")
print("-" * 70)
df_bronze_playlist = spark.sql(f"""
SELECT 
    10 as difficulty,
    1 as genre,
    0 as level,
    '{test_song_name}' as name,
    {test_song_id} as song_id,
    'Test Stepper' as stepauthor
""")
print("✓ Synthetic playlist record generated")
display(df_bronze_playlist)

# ===========================================================================
# SILVER LAYER
# ===========================================================================

print("\n" + "="*70)
print("[SILVER LAYER]")
print("=" * 70)

# Silver: notes-adjusted (decomposed notes from chart)
print("\n[4] silver__notes-adjusted")
print("-" * 70)
df_silver_notes = spark.sql(f"""
SELECT 
    {test_song_id} as song_id,
    1 as note_id,
    0.0 as timestamp,
    '0001' as orientation
UNION ALL
SELECT {test_song_id}, 2, 0.5, '0010'
UNION ALL
SELECT {test_song_id}, 3, 1.0, '0100'
UNION ALL
SELECT {test_song_id}, 4, 1.5, '1000'
UNION ALL
SELECT {test_song_id}, 5, 2.0, '1000'
UNION ALL
SELECT {test_song_id}, 5, 2.0, '0001'
""")
print("✓ Synthetic notes-adjusted records generated (6 decomposed notes)")
display(df_silver_notes)

# Silver: vertical-density
print("\n[5] silver__vertical-density")
print("-" * 70)
df_silver_vertical = spark.sql(f"""
SELECT 
    {test_song_id} as song_id,
    1 as note_id,
    0.0 as timestamp,
    '0001' as orientation,
    NULL as prev_timestamp,
    NULL as time_delta,
    0.0 as vertical_density,
    999999 as swf_version
UNION ALL
SELECT {test_song_id}, 2, 0.5, '0010', NULL, NULL, 0.0, 999999
UNION ALL
SELECT {test_song_id}, 3, 1.0, '0100', NULL, NULL, 0.0, 999999
UNION ALL
SELECT {test_song_id}, 4, 1.5, '1000', NULL, NULL, 0.0, 999999
UNION ALL
SELECT {test_song_id}, 5, 2.0, '1000', 1.5, 0.5, 2.0, 999999
UNION ALL
SELECT {test_song_id}, 5, 2.0, '0001', 0.0, 2.0, 0.5, 999999
""")
print("✓ Synthetic vertical-density records generated")
display(df_silver_vertical)

# Silver: horizontal-density
print("\n[6] silver__horizontal-density")
print("-" * 70)
df_silver_horizontal = spark.sql(f"""
SELECT 
    {test_song_id} as song_id,
    1 as note_id,
    0.0 as timestamp,
    '0001' as orientation,
    3.5 as horizontal_density,
    4 as nearby_notes_count,
    999999 as swf_version
UNION ALL
SELECT {test_song_id}, 2, 0.5, '0010', 4.0, 5, 999999
UNION ALL
SELECT {test_song_id}, 3, 1.0, '0100', 4.5, 5, 999999
UNION ALL
SELECT {test_song_id}, 4, 1.5, '1000', 4.0, 5, 999999
UNION ALL
SELECT {test_song_id}, 5, 2.0, '1000', 3.5, 4, 999999
UNION ALL
SELECT {test_song_id}, 5, 2.0, '0001', 3.5, 4, 999999
""")
print("✓ Synthetic horizontal-density records generated")
display(df_silver_horizontal)

# Silver: chart-features
print("\n[7] silver__chart-features")
print("-" * 70)
df_silver_chart_features = spark.sql(f"""
SELECT 
    {test_song_id} as song_id,
    5 as note_count,
    0.55 as song_length_log_normalized,
    999999 as swf_version
""")
print("✓ Synthetic chart-features record generated")
display(df_silver_chart_features)

# Silver: playlist
print("\n[8] silver__playlist")
print("-" * 70)
df_silver_playlist = spark.sql(f"""
SELECT 
    {test_song_id} as song_id,
    10 as difficulty,
    1 as genre,
    0 as level,
    '{test_song_name}' as name,
    'Test Stepper' as stepauthor
""")
print("✓ Synthetic silver__playlist record generated")
display(df_silver_playlist)

# ===========================================================================
# GOLD LAYER
# ===========================================================================

print("\n" + "="*70)
print("[GOLD LAYER]")
print("=" * 70)

# Gold: features
print("\n[9] gold__features")
print("-" * 70)
df_gold_features = spark.sql(f"""
SELECT 
    {test_song_id} as song_id,
    1 as note_id,
    '0001' as orientation,
    0.0 as vertical_density,
    3.5 as horizontal_density,
    0.0 as position_in_song,
    0.55 as song_length_log_normalized,
    0 as hold,
    1 as mine,
    '999999' as swf_version,
    10 as difficulty,
    0.85 as sample_weight
UNION ALL
SELECT {test_song_id}, 2, '0010', 0.0, 4.0, 0.25, 0.55, 0, 1, '999999', 10, 0.85
UNION ALL
SELECT {test_song_id}, 3, '0100', 0.0, 4.5, 0.5, 0.55, 0, 1, '999999', 10, 0.85
UNION ALL
SELECT {test_song_id}, 4, '1000', 0.0, 4.0, 0.75, 0.55, 0, 1, '999999', 10, 0.85
UNION ALL
SELECT {test_song_id}, 5, '1000', 2.0, 3.5, 1.0, 0.55, 0, 1, '999999', 10, 0.85
UNION ALL
SELECT {test_song_id}, 5, '0001', 0.5, 3.5, 1.0, 0.55, 0, 1, '999999', 10, 0.85
""")
print("✓ Synthetic gold__features records generated (6 feature vectors)")
display(df_gold_features)

# ===========================================================================
# SUMMARY
# ===========================================================================

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"✓ Generated synthetic data for ALL pipeline layers:")
print(f"  • Bronze: 3 tables (songlist, charts, playlist)")
print(f"  • Silver: 5 tables (notes-adjusted, vertical-density, horizontal-density, chart-features, playlist)")
print(f"  • Gold: 1 table (features)")
print(f"\nChart details:")
print(f"  • 5 notes in bronze__charts (multi-orientation included)")
print(f"  • 6 decomposed notes in silver (one note splits into 2 orientations)")
print(f"  • 6 feature vectors in gold (one per decomposed note)")
print(f"\nPurpose: Create complete dummy song ({test_song_id}) that exists in ALL tables")
print(f"         but NOT in the FFR API to test deletion detection")
print(f"\nNext steps:")
print(f"  1. Insert this data into all tables (see Cell 4)")
print(f"  2. Run bronze ingestion notebook")
print(f"  3. Verify song {test_song_id} is detected as deleted and removed from ALL layers")

In [0]:
# Verify that dummy song 99999 was deleted from all bronze, silver, and gold tables
from pyspark.sql.functions import col

test_song_id = 99999
test_song_name = "DUMMY_TEST_SONG"

print("="*70)
print("DELETION VERIFICATION - ALL INCREMENTAL TABLES")
print("="*70)
print(f"\nChecking if dummy song {test_song_id} was removed from all layers...\n")

# Define all tables to verify across bronze, silver, and gold layers
tables_to_verify = [
    # Bronze layer
    ("acubed.ffr.bronze__songlist", "id", test_song_id, "bronze"),
    ("acubed.ffr.bronze__charts", "song_id", test_song_id, "bronze"),
    ("acubed.ffr.bronze__playlist", "level", test_song_id, "bronze"),
    
    # Silver layer
    ("acubed.ffr.`silver__notes-adjusted`", "song_id", test_song_id, "silver"),
    ("acubed.ffr.`silver__vertical-density`", "song_id", test_song_id, "silver"),
    ("acubed.ffr.`silver__horizontal-density`", "song_id", test_song_id, "silver"),
    ("acubed.ffr.`silver__chart-features`", "song_id", test_song_id, "silver"),
    ("acubed.ffr.silver__playlist", "song_id", test_song_id, "silver"),
    
    # Gold layer
    ("acubed.ffr.`gold__features`", "song_id", test_song_id, "gold"),
]

success_count = 0
failure_count = 0
skipped_count = 0

layer_results = {"bronze": [], "silver": [], "gold": []}

for table_name, id_column, id_value, layer in tables_to_verify:
    short_name = table_name.replace('acubed.ffr.', '').replace('`', '')
    
    if spark.catalog.tableExists(table_name):
        test_song = spark.table(table_name).filter(col(id_column) == id_value)
        row_count = test_song.count()
        
        if row_count == 0:
            layer_results[layer].append((short_name, "✅ DELETED", True))
            success_count += 1
        else:
            layer_results[layer].append((short_name, f"❌ EXISTS ({row_count} rows)", False))
            failure_count += 1
    else:
        layer_results[layer].append((short_name, "⊘ table doesn't exist", None))
        skipped_count += 1

# Print results by layer
for layer_name in ["bronze", "silver", "gold"]:
    print(f"[{layer_name.upper()} LAYER]")
    print("-" * 70)
    
    if layer_results[layer_name]:
        for table, status, success in layer_results[layer_name]:
            print(f"{status:25} {table}")
    else:
        print("  No tables in this layer")
    print()

print("="*70)
print("VERIFICATION SUMMARY")
print("="*70)
print(f"\n✅ Successfully deleted: {success_count} tables")
print(f"❌ Failed to delete: {failure_count} tables")
print(f"⊘ Skipped (doesn't exist): {skipped_count} tables")

if failure_count == 0 and success_count > 0:
    print(f"\n✅ SUCCESS: Deletion propagated through entire pipeline!")
    print(f"\nDeletion workflow validated:")
    print(f"  1. Bronze ingestion detected song {test_song_id} not in API")
    print(f"  2. Removed from bronze__songlist via Delta DELETE")
    print(f"  3. Silver notebooks detected absence in bronze (via left join)")
    print(f"  4. Gold notebook detected absence in silver (via join)")
    print(f"  5. All {success_count} tables cleaned up successfully")
    
    print(f"\n📋 Verified Behaviors:")
    print(f"  • Bronze: API sync removes deleted songs")
    print(f"  • Silver: Incremental processing ignores missing bronze songs")
    print(f"  • Gold: Feature vectors exclude missing silver data")
    print(f"  • Delta DELETE: Tombstones created for audit trail")
    
    print(f"\n✅ TEST PASSED - Full pipeline deletion detection working correctly")
elif failure_count > 0:
    print(f"\n❌ TEST FAILED - Song still exists in {failure_count} tables")
    print(f"\nDiagnostics:")
    print(f"  • Run bronze ingestion: Did Cell 3 show 'Deleted songs: 1'?")
    print(f"  • Run silver notebooks: Did they process after bronze deletion?")
    print(f"  • Run gold notebook: Did it process after silver updates?")
    print(f"  • Check incremental logic: Are joins using left/inner correctly?")
    print(f"\nTroubleshooting:")
    print(f"  1. Verify bronze ingestion ran successfully")
    print(f"  2. Re-run each silver notebook in sequence")
    print(f"  3. Re-run gold notebook after silver completes")
    print(f"  4. Check notebook outputs for error messages")
else:
    print(f"\n⚠️ Could not verify - no tables to check")
    print(f"   Ensure pipeline tables exist before testing deletion")
    print(f"   Run Cell 2 to insert test data first")